# Four-class waste CNN experiment

Adapted from the retained course-project notebook. The architecture, augmentation and training logic are preserved. Local paths and saved outputs were removed. The historical validation matrix is recorded separately in `../reports/evaluation.json`.

Provide your own legally usable dataset under `data/train/<class>/` and `data/val/<class>/` with matching class folders. The original data and split manifest are unavailable. This notebook does not reproduce the historical result without them.


In [ ]:
from pathlib import Path
import os
DATA_ROOT = Path(os.environ.get("WASTE_DATA_ROOT", "../data"))
assert (DATA_ROOT / "train").is_dir(), "Set WASTE_DATA_ROOT to your prepared dataset"
assert (DATA_ROOT / "val").is_dir(), "Provide a separate validation folder"


In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Define image size and batch size
IMG_SIZE = (224, 224)
BATCH_SIZE = 16

# Data Preprocessing with Augmentation
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=10,          # Reduce rotation to 10 degrees
    width_shift_range=0.1,      # Reduce horizontal shift to 10%
    height_shift_range=0.1,     # Reduce vertical shift to 10%
    shear_range=0.1,            # Reduce shear to 10%
    zoom_range=0.1,             # Reduce zoom range to 10%
    horizontal_flip=True,
    fill_mode='nearest'
)


# Assuming your data is organized in categories (folders), e.g., 'plastic_bag', 'glass', etc.
train_dir = str(DATA_ROOT / "train")

# Load and process images
train_data = train_datagen.flow_from_directory(
    train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

In [ ]:
from tensorflow.keras import models, layers, regularizers

# Define the model with L1/L2 regularization and Batch Normalization
IMG_SIZE = 224  # Image size should be defined

model = models.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', padding='same',
                  kernel_regularizer=regularizers.l2(0.001), input_shape=(IMG_SIZE, IMG_SIZE, 3)),
    layers.BatchNormalization(),
    layers.MaxPooling2D(pool_size=(2, 2)),

    layers.Conv2D(64, (3, 3), activation='relu', padding='same',
                  kernel_regularizer=regularizers.l2(0.001)),
    layers.BatchNormalization(),
    layers.MaxPooling2D(pool_size=(2, 2)),

    layers.Conv2D(128, (3, 3), activation='relu', padding='same',
                  kernel_regularizer=regularizers.l2(0.001)),
    layers.BatchNormalization(),
    layers.MaxPooling2D(pool_size=(2, 2)),

    layers.Conv2D(128, (3, 3), activation='relu', padding='same',
                  kernel_regularizer=regularizers.l2(0.001)),
    layers.BatchNormalization(),
    layers.MaxPooling2D(pool_size=(2, 2)),
    
    # Flatten the 3D output to 1D for the Dense layer
    layers.Flatten(),

     # Dense layer with L1/L2 regularization
    layers.Dense(64, activation='relu', kernel_regularizer=regularizers.l2(0.001)),
    layers.BatchNormalization(),
    layers.Dropout(0.5),  # Dropout to further reduce overfitting

    # Dense layer with L1/L2 regularization
    layers.Dense(128, activation='relu', kernel_regularizer=regularizers.l2(0.001)),
    layers.BatchNormalization(),
    layers.Dropout(0.5),  # Dropout to further reduce overfitting

    # Dense layer with L1/L2 regularization
    layers.Dense(128, activation='relu', kernel_regularizer=regularizers.l2(0.001)),
    layers.BatchNormalization(),
    layers.Dropout(0.5),  # Dropout to further reduce overfitting
    
    # Output layer with softmax activation for multi-class classification
    layers.Dense(4, activation='softmax')
])

# Compile the model
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Summary of the model
model.summary()


In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# Set up callbacks (EarlyStopping and ModelCheckpoint)
early_stopping = EarlyStopping(monitor='loss', patience=5, restore_best_weights=True)
checkpoint = ModelCheckpoint('best_model_3.h5', save_best_only=True, monitor='loss', mode='min')

# Train the model
history = model.fit(
    train_data,
    epochs=50,
    callbacks=[early_stopping, checkpoint]
)

In [ ]:
from sklearn.metrics import confusion_matrix
import numpy as np
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import load_model

# Set parameters
IMG_SIZE = 224  # Image size (height and width)
BATCH_SIZE = 32  # Batch size for evaluation

# Load model (assuming the model has already been trained and saved)
# If you haven't saved your model, you can skip this or directly use the model you defined earlier.
# model = load_model('your_model_path.h5')  # Load the trained model

val_dir = str(DATA_ROOT / "val")

# Set up the ImageDataGenerator for validation data
val_datagen = ImageDataGenerator(rescale=1./255)

# Load validation data
val_data = val_datagen.flow_from_directory(
    val_dir,
    target_size=(IMG_SIZE, IMG_SIZE),  # Correct target_size as tuple (224, 224)
    batch_size=BATCH_SIZE,
    class_mode='categorical',  # For multi-class classification
    shuffle=False  # Important to keep the order of images for proper comparison
)

# Ensure that steps per epoch is calculated properly (considering incomplete last batch)
steps = val_data.samples // BATCH_SIZE + (1 if val_data.samples % BATCH_SIZE != 0 else 0)

# Get predictions from the model
y_pred = model.predict(val_data, steps=steps, verbose=1)  # Predict over all batches

# Convert predictions to class labels
y_pred_classes = np.argmax(y_pred, axis=1)

# Get the true labels
y_true = val_data.classes  # True labels

# Check if the number of predictions and true labels match
assert len(y_pred_classes) == len(y_true), f"Mismatch between predictions ({len(y_pred_classes)}) and true labels ({len(y_true)})!"

# Compute the confusion matrix
cm = confusion_matrix(y_true, y_pred_classes)

# Print confusion matrix
print("Confusion Matrix:\n", cm)



## Interpretation

The original training callbacks monitored training loss, not validation loss. The retained saved output reported 1,318 training images and 1,180 validation images, inconsistent with its intended 80/20 split. No split manifests were retained. Verify class mappings, group related images before splitting, and reserve an independent test set before drawing performance conclusions.

The original ROC example mixed four output classes with a five-class encoding and is omitted here. No ROC/AUC claim is made.
